<a href="https://colab.research.google.com/github/somendrew/Fine-tuning_LLMs/blob/main/Fine_Tuning_Lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q trl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 37.2 MB/s eta 0:00:00


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, pipeline, logging
from trl import SFTTrainer
from peft import LoraConfig, PeftModel, TaskType
import torch
import json
from datasets import Dataset

import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity(logging.CRITICAL)

In [4]:
with open("/content/science_10topics_100each.jsonl", "r") as f:
  data = [json.loads(line) for line in f if line.strip()]

# Convert to decoder-only format: input + output combined as one sequence
for d in data:
    output = d["output"]
    d["text"] = f"{d['input']}\n Question: {output['Question']}\n Answer: {output['Answer']}"

dataset = Dataset.from_list(data)

In [5]:
dataset

Dataset({
    features: ['input', 'output', 'text'],
    num_rows: 1000
})

In [6]:
dataset[1]

{'input': 'Generate a one word question on Photosynthesis',
 'output': {'Answer': 'Sunlight',
  'Question': 'What energy source drives photosynthesis?'},
 'text': 'Generate a one word question on Photosynthesis\n Question: What energy source drives photosynthesis?\n Answer: Sunlight'}

## Load Model and Tokenizer

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

#tokenizer
base_model = AutoModelForCausalLM.from_pretrained(

  MODEL_NAME,
  device_map="auto",
  torch_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(
  MODEL_NAME,
  trust_remote_code = True
  )

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_style = "right"


In [8]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

## BaseLine Generation

In [9]:
pipe = pipeline(
    'text-generation',
    model = base_model,
    tokenizer = tokenizer,
    max_length = 80
)

In [34]:
import json

topic = "Physics"

# 1. Improved prompt: More restrictive on the 'one word' rule
prompt = f"""Task: Generate a single factual {topic} question and answer.

Format:

  "Question": "..."/n,
  "Answer": "..."


"""

# 2. Call the pipeline
result = pipe(prompt, max_new_tokens=50, stop_sequence="}")

print(result[0]["generated_text"])

Task: Generate a single factual Physics question and answer.

Format:

  "Question": "..."/n,
  "Answer": "..."


Example:

Question: What are the properties of a gaseous fluid, and how do these properties affect its behavior when exposed to heat and pressure?

Answer: A gaseous fluid has two properties, viscosity and density


## LoRA Config

In [35]:
lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.01,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

### Preprocessing

In [36]:
def preprocess(sample):
  return sample['text']

## Training Config

In [37]:
import os
os.environ["WANDB_DISABLED"] = "True"

In [38]:
# Training Arguments
training_args = TrainingArguments(
    output_dir="./content",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=5e-5,
    logging_steps=50,
    save_strategy="epoch",
    report_to="none",
    fp16=True,
)

## Initialize Trainer

In [ ]:
# Trainer
trainer = SFTTrainer(
    model = base_model,
    train_dataset = dataset,
    peft_config = lora_config,
    formatting_func = preprocess,
    args = training_args,
)

## Start Training:

In [15]:
trainer.train()

{'loss': '2.883', 'grad_norm': '1.716', 'learning_rate': '4.347e-05', 'entropy': '2.404', 'num_tokens': '1.095e+04', 'mean_token_accuracy': '0.5013', 'epoch': '0.4'}
{'loss': '1.229', 'grad_norm': '0.9807', 'learning_rate': '3.68e-05', 'entropy': '1.271', 'num_tokens': '2.19e+04', 'mean_token_accuracy': '0.7497', 'epoch': '0.8'}
{'loss': '0.7751', 'grad_norm': '1.198', 'learning_rate': '3.013e-05', 'entropy': '0.8709', 'num_tokens': '3.284e+04', 'mean_token_accuracy': '0.8264', 'epoch': '1.2'}
{'loss': '0.66', 'grad_norm': '1.385', 'learning_rate': '2.347e-05', 'entropy': '0.7637', 'num_tokens': '4.381e+04', 'mean_token_accuracy': '0.8436', 'epoch': '1.6'}
{'loss': '0.6196', 'grad_norm': '1.25', 'learning_rate': '1.68e-05', 'entropy': '0.7259', 'num_tokens': '5.47e+04', 'mean_token_accuracy': '0.8461', 'epoch': '2'}
{'loss': '0.5836', 'grad_norm': '1.857', 'learning_rate': '1.013e-05', 'entropy': '0.6844', 'num_tokens': '6.566e+04', 'mean_token_accuracy': '0.8552', 'epoch': '2.4'}
{'lo

TrainOutput(global_step=375, training_loss=1.018468780517578, metrics={'train_runtime': 196.852, 'train_samples_per_second': 15.24, 'train_steps_per_second': 1.905, 'total_flos': 536204158402560.0, 'train_loss': 1.018468780517578})

## Save the fine-tuned Model

In [16]:
new_model_name = "tinyllama_finetuned_lora"

trainer.model.save_pretrained(new_model_name)
tokenizer.save_pretrained(new_model_name)

('tinyllama_finetuned_lora/tokenizer_config.json',
 'tinyllama_finetuned_lora/chat_template.jinja',
 'tinyllama_finetuned_lora/tokenizer.json')

In [42]:
!unzip my_project.zip

Archive:  my_project.zip
replace content/tinyllama_finetuned_lora/README.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: content/tinyllama_finetuned_lora/README.md  
  inflating: content/tinyllama_finetuned_lora/tokenizer.json  
  inflating: content/tinyllama_finetuned_lora/adapter_config.json  
  inflating: content/tinyllama_finetuned_lora/tokenizer_config.json  
  inflating: content/tinyllama_finetuned_lora/chat_template.jinja  
  inflating: content/tinyllama_finetuned_lora/adapter_model.safetensors  


In [ ]:
new_model_name = "/content/content/tinyllama_finetuned_lora"

pipe = pipeline(task="text-generation", model=new_model_name, tokenizer=new_model_name, max_length=80)

## Inference on test Prompt

In [52]:
topic = "Photosynthesis"

prompt = f"""Task: Generate a single factual {topic} question and answer.

Format:

  "Question": "..."/n,
  "Answer": "..."

"""

prompt = f"""Generate a question and answer about {topic}
Return the result in JSON format."""

result = pipe(prompt, max_new_tokens=50, stop_sequence="}")

In [53]:
print(result[0]['generated_text'])

Generate a question and answer about Photosynthesis
Return the result in JSON format. 
<|assistant|>
Question: What process is responsible for producing food from sunlight using photosynthesis?

Answer: The process of photosynthesis is responsible for producing food (glucose) from sunlight


## Merge LoRA adapters with base model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # base model
lora_model_path = "/content/content/tinyllama_finetuned_lora"  # directory with adapter weights

# Load base model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(base_model_name)

# Load LoRA adapter on top of base model
model = PeftModel.from_pretrained(base_model, lora_model_path)

In [55]:
# merge both models
merged_model = model.merge_and_unload()

In [56]:
from transformers import pipeline

pipe = pipeline(
    task="text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    max_length=80
)

In [57]:
prompt = "Generate a question on Electricity"
result = pipe(prompt)
print(result[0]['generated_text'])

Generate a question on Electricity
 Question: What charge flows through a conductor?
 Answer: Current
